# GPU Final Notebook - Capitals

Modernized final capitals run using behavior-specific capitals SAEs plus contrast-target attribution graphs, all-position full latent swaps, graph-positive sparse swaps, and separate `capitals_final_*` outputs.

This notebook trains or resumes final capitals-specific SAEs in `sae_checkpoints_capitals_final_train`, then saves distinct `capitals_final_*` graph/intervention outputs.


## Step 1 - Mount Drive and fetch code


In [1]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary source for code. Drive project.zip is kept as a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "INSERT REPO URL.git"
repo_dir = "/content/test_run"
use_github_code = True

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
if use_github_code:
    try:
        clone_dir = repo_dir
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        else:
            if os.path.exists(clone_dir) and os.listdir(clone_dir):
                clone_dir = "/content/test_run_github"
            if os.path.isdir(os.path.join(clone_dir, ".git")):
                run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
            elif os.path.exists(clone_dir) and os.listdir(clone_dir):
                raise RuntimeError(f"{clone_dir} exists but is not a git repo")
            else:
                run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
        os.chdir(clone_dir)
        github_ok = True
        print(f"Using GitHub checkout: {os.getcwd()}")
    except Exception as exc:
        print("GitHub checkout failed; falling back to Drive project.zip.")
        print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


Mounted at /content/drive
$ git -C /content/test_run pull --ff-only
Using GitHub checkout: /content/test_run
Current working directory: /content/test_run
total 780
-rw-r--r-- 1 root root   6680 Jul 10 04:34 ai_handoff_summary.txt
-rw-r--r-- 1 root root   8894 Jul 10 04:34 changes_summary.txt
drwxr-xr-x 2 root root   4096 Jul 10 04:34 configs
drwxr-xr-x 2 root root   4096 Jul 10 04:34 data
-rw-r--r-- 1 root root    264 Jul 10 04:34 environment.yml
-rw-r--r-- 1 root root  19186 Jul 10 04:34 IMPLEMENTATION_PLAN.md
-rw-r--r-- 1 root root   1579 Jul 10 04:34 Instructions.md
drwxr-xr-x 4 root root   4096 Jul 10 04:36 mechanistic_data
drwxr-xr-x 2 root root   4096 Jul 10 04:36 mechanistic_data_math_final_train
drwxr-xr-x 2 root root   4096 Jul 10 06:12 mechanistic_interpretability_repro.egg-info
drwxr-xr-x 3 root root   4096 Jul 10 06:23 outputs
-rw-r--r-- 1 root root   6551 Jul 10 04:34 project.txt
-rw-r--r-- 1 root root  11435 Jul 10 04:34 README.md
-rw-r--r-- 1 root root  12564 Jul 10 04:3

## Step 2 - Install dependencies


In [2]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


Obtaining file:///content/test_run
  Preparing metadata (setup.py) ... done
  Attempting uninstall: mechanistic-interpretability-repro
    Found existing installation: mechanistic-interpretability-repro 0.1.0
    Uninstalling mechanistic-interpretability-repro-0.1.0:
      Successfully uninstalled mechanistic-interpretability-repro-0.1.0
  Running setup.py develop for mechanistic-interpretability-repro


## Step 3 - Generate capitals dataset


In [3]:
# Step 3: Generate capitals dataset
!python data/generate_datasets.py --capitals
print("Dataset files:")
!ls -lh data/capitals_data.csv || true
!ls -lh data/*capital* || true


Generating Datasets...

Flag '--capitals' captured. Building structural sentences for geography data...
Saved 1000 capital entries with text templates.
Standalone Mode complete:
Saved 1000 addition problems and 
Saved 1000 unit problems.
Dataset files:
-rw-r--r-- 1 root root 97K Jul 10 06:27 data/capitals_data.csv
-rw-r--r-- 1 root root 6.3K Jul 10 04:34 data/capitals_base.csv
-rw-r--r-- 1 root root  97K Jul 10 06:27 data/capitals_data.csv


## Step 4 - Restore final capitals SAE checkpoints from Drive if available


In [4]:
# Restore final capitals-specific SAE checkpoints from Drive if they already exist.
# If this succeeds, the capture/training cells below will skip themselves.
from pathlib import Path
import shutil

FINAL_LAYERS = [4, 8, 12, 16, 20, 24, 28]
local_sae_dir = Path('/content/test_run/mechanistic_data/sae_checkpoints_capitals_final_train')
drive_sae_dir = Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train')

def has_complete_sae_set(path, layers):
    path = Path(path)
    return all(
        (path / f'sae_layer{layer}.pt').exists()
        and (path / f'sae_layer{layer}_metadata.json').exists()
        and (path / f'latents_layer{layer}.npy').exists()
        for layer in layers
    )

CAPITALS_FINAL_SAES_READY = False
if has_complete_sae_set(drive_sae_dir, FINAL_LAYERS):
    local_sae_dir.mkdir(parents=True, exist_ok=True)
    for src in sorted(drive_sae_dir.iterdir()):
        if src.is_file():
            shutil.copy2(src, local_sae_dir / src.name)
    CAPITALS_FINAL_SAES_READY = has_complete_sae_set(local_sae_dir, FINAL_LAYERS)
    print(f'Restored complete final capitals SAE checkpoint set from Drive: {drive_sae_dir}')
    print(f'Local checkpoint dir: {local_sae_dir}')
else:
    print(f'No complete final capitals SAE checkpoint set found at {drive_sae_dir}.')
    print('The notebook will capture activations and train SAEs below.')

print('CAPITALS_FINAL_SAES_READY =', CAPITALS_FINAL_SAES_READY)


No complete final capitals SAE checkpoint set found at /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train.
The notebook will capture activations and train SAEs below.
CAPITALS_FINAL_SAES_READY = False


## Step 5 - Capture capitals-only activations


In [5]:
# Capture capitals-only MLP activations for final SAE training, unless checkpoints were restored.
import subprocess

if globals().get('CAPITALS_FINAL_SAES_READY', False):
    print('Skipping activation capture because final capitals SAE checkpoints were restored from Drive.')
else:
    subprocess.run([
        'python', 'src/capture_activations.py',
        '--output-dir', 'mechanistic_data_capitals_final_train',
        '--behaviours', 'capitals',
        '--layers', '4', '8', '12', '16', '20', '24', '28',
        '--seed', '787',
    ], check=True)
    print('Final capitals activation bundle:')
    subprocess.run(['ls', '-lh', 'mechanistic_data_capitals_final_train'], check=True)


Final capitals activation bundle:


## Step 6 - Train final capitals-specific SAEs


In [6]:
# Train final capitals-specific SAEs for all selected layers, unless checkpoints were restored.
import subprocess

if globals().get('CAPITALS_FINAL_SAES_READY', False):
    print('Skipping SAE training because final capitals SAE checkpoints were restored from Drive.')
else:
    subprocess.run([
        'python', 'src/train.py',
        '--config', 'configs/sae_capitals_final_train_config.yaml',
        '--drive-dir', '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train',
    ], check=True)

print('Final capitals local checkpoints:')
subprocess.run(['ls', '-lh', 'mechanistic_data/sae_checkpoints_capitals_final_train'], check=True)
print('Final capitals Drive checkpoints:')
subprocess.run(['ls', '-lh', '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train'], check=True)


Final capitals local checkpoints:
Final capitals Drive checkpoints:


CompletedProcess(args=['ls', '-lh', '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train'], returncode=0)

## Step 7 - Sanity check the Dallas/Oakland minimal pair


In [7]:
# Confirm token positions before all-position swaps.
# Dallas -> Austin and Oakland -> Sacramento should be comparable prompts.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --print-tokens

!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --print-tokens


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.74it/s]
Prompt token positions:
   0: id=17417    token='Fact'
   1: id=25       token=':'
   2: id=576      token=' The'
   3: id=6722     token=' capital'
   4: id=315      token=' of'
   5: id=279      token=' the'
   6: id=1584     token=' state'
   7: id=8482     token=' containing'
   8: id=18542    token=' Dallas'
   9: id=374      token=' is'
  10: id=6941     token=' named'
Use --positions last, --positions all, or comma-separated indices such as --positions 4,5,9
Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.18it/s]
Prompt token positions:
   0: id=17417    token='Fact'
   1: id=25       token=':'
   2: id=576      token=' The'
   3: id=6722     token=' capita

## Step 8 - Rebuild contrast attribution graphs


In [8]:
# Dallas graph: Austin vs Sacramento.
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target "Austin" \
  --contrast-target "Sacramento" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/capitals_final_dallas_austin_vs_sacramento_graph.json \
  --output-html outputs/capitals_final_dallas_austin_vs_sacramento_graph.html \
  --output-mermaid outputs/capitals_final_dallas_austin_vs_sacramento_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.27it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_capitals_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1531)
Loaded SAE for layer 8 (scaling factor: 0.3549)
Loaded SAE for layer 12 (scaling factor: 0.2894)
Loaded SAE for layer 16 (scaling factor: 0.2888)
Loaded SAE for layer 20 (scaling factor: 0.3368)
Loaded SAE for layer 24 (scaling factor: 0.5745)
Loaded SAE for layer 28 (scaling factor: 0.9752)
Running forward pass...

Top 5 model predictions:
  ' Austin' (prob: 0.2773, logit: 16.50)
  ' Dallas' (prob: 0.2158, logit: 16.25)
  ' Fort' (prob: 0.1230, logit: 15.69)
  ' after' (prob: 0.1230, logit: 15.69)
  ' what' (prob: 0.0659, logit: 15.06)

Target to

In [9]:
# Oakland graph: Sacramento vs Austin.
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --target "Sacramento" \
  --contrast-target "Austin" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/capitals_final_oakland_sacramento_vs_austin_graph.json \
  --output-html outputs/capitals_final_oakland_sacramento_vs_austin_graph.html \
  --output-mermaid outputs/capitals_final_oakland_sacramento_vs_austin_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 184.36it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_capitals_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1531)
Loaded SAE for layer 8 (scaling factor: 0.3549)
Loaded SAE for layer 12 (scaling factor: 0.2894)
Loaded SAE for layer 16 (scaling factor: 0.2888)
Loaded SAE for layer 20 (scaling factor: 0.3368)
Loaded SAE for layer 24 (scaling factor: 0.5745)
Loaded SAE for layer 28 (scaling factor: 0.9752)
Running forward pass...

Top 5 model predictions:
  ' after' (prob: 0.4141, logit: 18.50)
  ' Sacramento' (prob: 0.2217, logit: 17.88)
  ' Oakland' (prob: 0.1348, logit: 17.38)
  ' San' (prob: 0.1348, logit: 17.38)
  ' what' (prob: 0.0249, logit: 15.69)

Targe

In [10]:
# Persist final capitals graphs immediately.
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/capitals_final_*_graph.*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


Copied capitals_final_oakland_sacramento_vs_austin_graph.json
Copied capitals_final_oakland_sacramento_vs_austin_graph.html
Copied capitals_final_dallas_austin_vs_sacramento_graph.json
Copied capitals_final_dallas_austin_vs_sacramento_graph.md
Copied capitals_final_dallas_austin_vs_sacramento_graph.html
Copied capitals_final_oakland_sacramento_vs_austin_graph.md
Done!


## Step 9 - Component and sparse controls


In [11]:
# All-position MLP knockout on Dallas/Austin.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin, Sacramento" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --full-knockout \
  --knockout-component mlp \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_knockout_mlp_allpos_dallas.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.40it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2773, logit: 16.50)
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target ' Dallas': prob: 0.2158, logit: 16.25
  - Target 'Sac': prob: 0.0000, logit: -6.38
  - Target ' Sacramento': prob: 0.0000, logit: 2.67

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]: {}...
Editing token positions:
   0: id=17417    token='Fact'
   1: id=25       token=':'
   2: id=576      token=' The'
   3: id=6722     token=' capital'
   4: id=315      token=' of'
   5: id=279      token=' the'
   6: id=1584     token=' state'
   7: id=8482     t

In [12]:
# Progressive sparse ablation of Dallas graph-positive features.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin, Sacramento" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_dallas_austin_vs_sacramento_graph.json \
  --graph-feature-sign positive \
  --scan \
  --positions all \
  --output outputs/capitals_final_scan_dallas_positive_allpos.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 184.85it/s]
Loaded 152 nodes from graph JSON; extracted features across 7 layers (61 total features)
  Graph feature sign filter: positive (skipped 79 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2773, logit: 16.50)
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target ' Dallas': prob: 0.2158, logit: 16.25
  - Target 'Sac': prob: 0.0000, logit: -6.38
  - Target ' Sacramento': prob: 0.0000, logit: 2.67

[2/3] Running SAE Reconstruction-only Baseline (no features inhibited)...
Running forward pass with inhibition at positions [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]: {}...
Editing token positions:
   0: id=17417    token='Fact'
   1: id=25       token=':'
   2: id=576      token='

## Step 10 - Full latent minimal-pair swaps


In [13]:
# Patch Oakland/Sacramento activations into Dallas/Austin.
# Desired direction: Austin falls and Sacramento rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --target-token "Austin, Sacramento" \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_swap_full_oakland_to_dallas.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.15it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2773, logit: 16.50)
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Dallas': prob: 0.2158, logit: 16.25
  - Target 'Sac': prob: 0.0000, logit: -6.38
  - Target ' Sacramento': prob: 0.0000, logit: 2.67
  - Target ' San': prob: 0.0000, logit: 7.69

Source baseline prediction on 'Fact: The capital of the state containing Oakland is named':
  - Target ' after': prob: 0.4141, logit: 18.50
  - Target ' Austin': prob: 0.0000, logit: 7.34
  - Target 'Austin': prob: 0.0000, logit: -0.84
  - Target ' Dallas': prob: 0.0000, logit: 5.88
  - Target 'Sac': prob: 0.0000, logit: 7.19
  - Target ' Sacramento': prob: 0.2217, logit: 17.88
  -

In [14]:
# Reverse direction: patch Dallas/Austin activations into Oakland/Sacramento.
# Desired direction: Sacramento falls and Austin rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Dallas is named" \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --target-token "Austin, Sacramento" \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_swap_full_dallas_to_oakland.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.06it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' after' (prob: 0.4141, logit: 18.50)
  - Target ' after': prob: 0.4141, logit: 18.50
  - Target 'Austin': prob: 0.0000, logit: -0.84
  - Target ' Austin': prob: 0.0000, logit: 7.34
  - Target ' Dallas': prob: 0.0000, logit: 5.88
  - Target 'Sac': prob: 0.0000, logit: 7.19
  - Target ' Sacramento': prob: 0.2217, logit: 17.88
  - Target ' San': prob: 0.1348, logit: 17.38

Source baseline prediction on 'Fact: The capital of the state containing Dallas is named':
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target ' Dallas': prob: 0.2158, logit: 16.25
  - Target 'Sac': prob: 0.0000, logit: -6.38
  - Target ' Sacramento': prob: 0.0000, logit: 2.67
  - 

## Step 11 - Sparse graph-feature swaps


In [15]:
# Sparse Oakland -> Dallas, using Oakland graph-positive Sacramento features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_oakland_sacramento_vs_austin_graph.json \
  --graph-feature-sign positive \
  --target-token "Austin, Sacramento" \
  --positions all \
  --output outputs/capitals_final_swap_sparse_oakland_to_dallas.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 187.06it/s]
Loaded 152 nodes from graph JSON; extracted features across 6 layers (61 total features)
  Graph feature sign filter: positive (skipped 79 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: ' Austin' (prob: 0.2773, logit: 16.50)
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target 'Austin': prob: 0.0000, logit: 7.56
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target ' Dallas': prob: 0.2158, logit: 16.25
  - Target 'Sac': prob: 0.0000, logit: -6.38
  - Target ' Sacramento': prob: 0.0000, logit: 2.67
  - Target ' San': prob: 0.0000, logit: 7.69

Source baseline prediction on 'Fact: The capital of the state containing Oakland is named':
  - Target ' after': prob: 0.4141, logit: 18.50
  - Target 'Austin': prob: 0.0000, logit: -0.84
  - Target ' Austin': prob: 0.0000, logit:

In [16]:
# Sparse Dallas -> Oakland, using Dallas graph-positive Austin features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Dallas is named" \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_dallas_austin_vs_sacramento_graph.json \
  --graph-feature-sign positive \
  --target-token "Austin, Sacramento" \
  --positions all \
  --output outputs/capitals_final_swap_sparse_dallas_to_oakland.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.77it/s]
Loaded 152 nodes from graph JSON; extracted features across 7 layers (61 total features)
  Graph feature sign filter: positive (skipped 79 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: ' after' (prob: 0.4141, logit: 18.50)
  - Target ' after': prob: 0.4141, logit: 18.50
  - Target ' Austin': prob: 0.0000, logit: 7.34
  - Target 'Austin': prob: 0.0000, logit: -0.84
  - Target ' Dallas': prob: 0.0000, logit: 5.88
  - Target 'Sac': prob: 0.0000, logit: 7.19
  - Target ' Sacramento': prob: 0.2217, logit: 17.88
  - Target ' San': prob: 0.1348, logit: 17.38

Source baseline prediction on 'Fact: The capital of the state containing Dallas is named':
  - Target ' after': prob: 0.1230, logit: 15.69
  - Target ' Austin': prob: 0.2773, logit: 16.50
  - Target 'Austin': prob: 0.0000, logit: 7

## Step 12 - High-confidence capitals control: Zarqa/Amman and Basra/Baghdad
These examples had much stronger baseline probabilities than Dallas/Oakland in the baseline scan. Use this section to test whether the weak capitals interventions are mostly caused by low-confidence prompts rather than the intervention machinery.


In [18]:
# Token sanity check for the high-confidence capitals pair.
# Read the printed tokens before running expensive graph cells. If the prompts tokenize very differently,
# interpret all-position swaps cautiously.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Zarqa is named" \
  --target-token "Amman, Baghdad" \
  --print-tokens

!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Basra is named" \
  --target-token "Amman, Baghdad" \
  --print-tokens


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.30it/s]
Prompt token positions:
   0: id=17417    token='Fact'
   1: id=25       token=':'
   2: id=576      token=' The'
   3: id=6722     token=' capital'
   4: id=315      token=' of'
   5: id=279      token=' the'
   6: id=3146     token=' country'
   7: id=8482     token=' containing'
   8: id=61431    token=' Zar'
   9: id=15445    token='qa'
  10: id=374      token=' is'
  11: id=6941     token=' named'
Use --positions last, --positions all, or comma-separated indices such as --positions 4,5,9
Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.72it/s]
Prompt token positions:
   0: id=17417    token='Fact'
   1: id=25       token=':'
   2: id=576      token=' The'
   3

In [19]:
# Zarqa graph: Amman vs Baghdad.
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the country containing Zarqa is named" \
  --target "Amman" \
  --contrast-target "Baghdad" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/capitals_final_hiconf_zarqa_amman_vs_baghdad_graph.json \
  --output-html outputs/capitals_final_hiconf_zarqa_amman_vs_baghdad_graph.html \
  --output-mermaid outputs/capitals_final_hiconf_zarqa_amman_vs_baghdad_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 185.06it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_capitals_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1531)
Loaded SAE for layer 8 (scaling factor: 0.3549)
Loaded SAE for layer 12 (scaling factor: 0.2894)
Loaded SAE for layer 16 (scaling factor: 0.2888)
Loaded SAE for layer 20 (scaling factor: 0.3368)
Loaded SAE for layer 24 (scaling factor: 0.5745)
Loaded SAE for layer 28 (scaling factor: 0.9752)
Running forward pass...

Top 5 model predictions:
  ' Am' (prob: 0.9844, logit: 21.25)
  ' A' (prob: 0.0033, logit: 15.56)
  ' as' (prob: 0.0024, logit: 15.25)
  ' Zar' (prob: 0.0019, logit: 15.00)
  ' I' (prob: 0.0010, logit: 14.31)

Target token for attribut

In [20]:
# Basra graph: Baghdad vs Amman.
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the country containing Basra is named" \
  --target "Baghdad" \
  --contrast-target "Amman" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/capitals_final_hiconf_basra_baghdad_vs_amman_graph.json \
  --output-html outputs/capitals_final_hiconf_basra_baghdad_vs_amman_graph.html \
  --output-mermaid outputs/capitals_final_hiconf_basra_baghdad_vs_amman_graph.md


Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 181.83it/s]
Disabling gradients for all model parameters for speed and memory efficiency.
Loading SAE models from /content/test_run/mechanistic_data/sae_checkpoints_capitals_final_train to device: cuda...
Loaded SAE for layer 4 (scaling factor: 0.1531)
Loaded SAE for layer 8 (scaling factor: 0.3549)
Loaded SAE for layer 12 (scaling factor: 0.2894)
Loaded SAE for layer 16 (scaling factor: 0.2888)
Loaded SAE for layer 20 (scaling factor: 0.3368)
Loaded SAE for layer 24 (scaling factor: 0.5745)
Loaded SAE for layer 28 (scaling factor: 0.9752)
Running forward pass...

Top 5 model predictions:
  ' Baghdad' (prob: 0.9648, logit: 19.38)
  ' as' (prob: 0.0084, logit: 14.62)
  ' after' (prob: 0.0069, logit: 14.44)
  ' Iraq' (prob: 0.0029, logit: 13.56)
  ' "' (prob: 0.0025, logit: 13.44)

Target token fo

In [21]:
# Full latent swap: Basra/Baghdad -> Zarqa/Amman.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the country containing Basra is named" \
  --prompt "Fact: The capital of the country containing Zarqa is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --target-token "Amman, Baghdad" \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_hiconf_swap_full_basra_to_zarqa.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 180.35it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Am' (prob: 0.9844, logit: 21.25)
  - Target ' A': prob: 0.0033, logit: 15.56
  - Target ' after': prob: 0.0006, logit: 13.81
  - Target ' Am': prob: 0.9844, logit: 21.25
  - Target 'Am': prob: 0.0000, logit: 11.06
  - Target ' as': prob: 0.0024, logit: 15.25
  - Target 'Bag': prob: 0.0000, logit: -3.73
  - Target ' Baghdad': prob: 0.0000, logit: 6.97

Source baseline prediction on 'Fact: The capital of the country containing Basra is named':
  - Target ' A': prob: 0.0000, logit: 6.34
  - Target ' after': prob: 0.0069, logit: 14.44
  - Target ' Am': prob: 0.0001, logit: 9.62
  - Target 'Am': prob: 0.0000, logit: 1.38
  - Target ' as': prob: 0.0084, logit: 14.62
  - Target 'Bag': prob: 0.0000, logit: 6.72
  - Target ' Baghdad': prob: 0.9648, logit: 

In [22]:
# Full latent reverse: Zarqa/Amman -> Basra/Baghdad.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the country containing Zarqa is named" \
  --prompt "Fact: The capital of the country containing Basra is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --target-token "Amman, Baghdad" \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_hiconf_swap_full_zarqa_to_basra.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 187.28it/s]

[1/3] Clean Model Baseline:
  - Top prediction: ' Baghdad' (prob: 0.9648, logit: 19.38)
  - Target ' A': prob: 0.0000, logit: 6.34
  - Target ' after': prob: 0.0069, logit: 14.44
  - Target ' Am': prob: 0.0001, logit: 9.62
  - Target 'Am': prob: 0.0000, logit: 1.38
  - Target ' as': prob: 0.0084, logit: 14.62
  - Target 'Bag': prob: 0.0000, logit: 6.72
  - Target ' Baghdad': prob: 0.9648, logit: 19.38

Source baseline prediction on 'Fact: The capital of the country containing Zarqa is named':
  - Target ' A': prob: 0.0033, logit: 15.56
  - Target ' after': prob: 0.0006, logit: 13.81
  - Target ' Am': prob: 0.9844, logit: 21.25
  - Target 'Am': prob: 0.0000, logit: 11.06
  - Target ' as': prob: 0.0024, logit: 15.25
  - Target 'Bag': prob: 0.0000, logit: -3.73
  - Target ' Baghdad': prob: 0.0000, l

In [23]:
# Sparse Basra -> Zarqa, using Basra graph-positive Baghdad features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the country containing Basra is named" \
  --prompt "Fact: The capital of the country containing Zarqa is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_hiconf_basra_baghdad_vs_amman_graph.json \
  --graph-feature-sign positive \
  --target-token "Amman, Baghdad" \
  --positions all \
  --output outputs/capitals_final_hiconf_swap_sparse_basra_to_zarqa.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 186.50it/s]
Loaded 153 nodes from graph JSON; extracted features across 7 layers (80 total features)
  Graph feature sign filter: positive (skipped 60 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: ' Am' (prob: 0.9844, logit: 21.25)
  - Target ' A': prob: 0.0033, logit: 15.56
  - Target ' after': prob: 0.0006, logit: 13.81
  - Target ' Am': prob: 0.9844, logit: 21.25
  - Target 'Am': prob: 0.0000, logit: 11.06
  - Target ' as': prob: 0.0024, logit: 15.25
  - Target 'Bag': prob: 0.0000, logit: -3.73
  - Target ' Baghdad': prob: 0.0000, logit: 6.97

Source baseline prediction on 'Fact: The capital of the country containing Basra is named':
  - Target ' A': prob: 0.0000, logit: 6.34
  - Target ' after': prob: 0.0069, logit: 14.44
  - Target ' Am': prob: 0.0001, logit: 9.62
  - Target 'Am': prob:

In [24]:
# Sparse Zarqa -> Basra, using Zarqa graph-positive Amman features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the country containing Zarqa is named" \
  --prompt "Fact: The capital of the country containing Basra is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_hiconf_zarqa_amman_vs_baghdad_graph.json \
  --graph-feature-sign positive \
  --target-token "Amman, Baghdad" \
  --positions all \
  --output outputs/capitals_final_hiconf_swap_sparse_zarqa_to_basra.json


Loading model...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
Loading weights: 100% 398/398 [00:02<00:00, 184.17it/s]
Loaded 153 nodes from graph JSON; extracted features across 7 layers (76 total features)
  Graph feature sign filter: positive (skipped 64 feature nodes)

[1/3] Clean Model Baseline:
  - Top prediction: ' Baghdad' (prob: 0.9648, logit: 19.38)
  - Target ' A': prob: 0.0000, logit: 6.34
  - Target ' after': prob: 0.0069, logit: 14.44
  - Target ' Am': prob: 0.0001, logit: 9.62
  - Target 'Am': prob: 0.0000, logit: 1.38
  - Target ' as': prob: 0.0084, logit: 14.62
  - Target 'Bag': prob: 0.0000, logit: 6.72
  - Target ' Baghdad': prob: 0.9648, logit: 19.38

Source baseline prediction on 'Fact: The capital of the country containing Zarqa is named':
  - Target ' A': prob: 0.0033, logit: 15.56
  - Target ' after': prob: 0.0006, logit: 13.81
  - Target ' Am': prob: 0.9844, logit: 21.25
  - Target 'Am': p

## Step 13 - Persist final capitals outputs


In [25]:
# Copy final capitals graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/capitals_final_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


Copied capitals_final_knockout_mlp_allpos_dallas.json
Copied capitals_final_hiconf_swap_sparse_zarqa_to_basra.json
Copied capitals_final_swap_sparse_dallas_to_oakland.json
Copied capitals_final_oakland_sacramento_vs_austin_graph.json
Copied capitals_final_oakland_sacramento_vs_austin_graph.html
Copied capitals_final_dallas_austin_vs_sacramento_graph.json
Copied capitals_final_hiconf_swap_sparse_basra_to_zarqa.json
Copied capitals_final_dallas_austin_vs_sacramento_graph.md
Copied capitals_final_hiconf_basra_baghdad_vs_amman_graph.json
Copied capitals_final_swap_full_dallas_to_oakland.json
Copied capitals_final_hiconf_basra_baghdad_vs_amman_graph.html
Copied capitals_final_dallas_austin_vs_sacramento_graph.html
Copied capitals_final_swap_sparse_oakland_to_dallas.json
Copied capitals_final_hiconf_zarqa_amman_vs_baghdad_graph.md
Copied capitals_final_scan_dallas_positive_allpos.json
Copied capitals_final_hiconf_swap_full_basra_to_zarqa.json
Copied capitals_final_swap_full_oakland_to_dallas